In [1]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [2]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, num_outputs)
        )

    def forward(self, x):
        return self.layers(x)

In [3]:
class ToyDataset(Dataset):
    def __getitem__(self, index):
        return self.features[index], self.labels[index]
    def __init__(self, X, y):
        self.features = X
        self.labels = y
    def __len__(self):
        return len(self.features)

In [4]:
X_train = torch.tensor(
    [
        [-1.2, 3.1],
        [-0.9, 2.9],
        [-0.5, 2.6],
        [2.3, -1.1],
        [2.7, -1.5]
    ]
)
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor(
    [
        [-0.8, 2.8],
        [2.6, -1.6]
    ]
)
y_test = torch.tensor([0, 1])

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [5]:
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False)

In [6]:
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
num_epochs = 3

for epoch in range(num_epochs):
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d} | Batch {batch_idx:03d}/{len(train_loader):03d} | Train/Val Loss: {loss:.2f}")

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.61
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 1.69
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.01
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.00
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.00
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


In [39]:
def compute_accuracy(model, dataloader):
    model.eval()
    correct = 0
    for features, labels in dataloader:
        with torch.no_grad():
            logits = model(features)
            predictions = torch.argmax(logits, dim=1)
            correct += torch.eq(predictions, labels).sum().float().item()
    return correct / len(dataloader.dataset)

compute_accuracy(model, test_loader)

1.0

In [41]:
torch.save(model.state_dict(), "model.pth")

In [44]:
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [45]:
model

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [46]:
model.layers

Sequential(
  (0): Linear(in_features=2, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=2, bias=True)
)

In [53]:
parameters_num = [i.numel() for i in model.parameters()]
sum(parameters_num)
    

642